<a href="https://colab.research.google.com/github/arvinmungara-alt/Agentic-AI/blob/main/AI_Agent_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langgraph langchain langchain-google-genai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.4 MB/s eta 0:00:00


In [ ]:
import os
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6IU6jUe8GnDX6tIR6w05Tt7qpHw0uuFhQXlm8z9EmUvrg"  # paste your real key here

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

In [ ]:
class AgentState(TypedDict):
    topic: str
    research_notes: str
    current_draft: str
    reviewer_feedback: str
    approved: bool

In [ ]:
def research_node(state):
    prompt = f"You are an HHS Epidemiologist. Research and list bullet-point facts about: {state['topic']}. Include outbreak data, symptoms, and transmission info."
    response = llm.invoke(prompt)
    return {"research_notes": response.content}

def writer_node(state):
    prompt = f"""You are a Public Health Communications Officer.
Using these research facts:
{state['research_notes']}

Write an official FDA/HHS-style public health advisory memo about {state['topic']}."""
    response = llm.invoke(prompt)
    return {"current_draft": response.content}

def reviewer_node(state):
    prompt = f"""You are a Compliance Officer. Check this draft against the research facts for accuracy.
Research: {state['research_notes']}
Draft: {state['current_draft']}
List any errors, or say 'APPROVED' if accurate."""
    response = llm.invoke(prompt)
    return {"reviewer_feedback": response.content}

def human_approval_node(state):
    return {}  # placeholder — execution pauses here until a human resumes it

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("Researcher", research_node)
workflow.add_node("Writer", writer_node)
workflow.add_node("Reviewer", reviewer_node)
workflow.add_node("Human_Approval", human_approval_node)

workflow.set_entry_point("Researcher")
workflow.add_edge("Researcher", "Writer")
workflow.add_edge("Writer", "Reviewer")
workflow.add_edge("Reviewer", "Human_Approval")
workflow.add_edge("Human_Approval", END)

In [ ]:
memory = MemorySaver()
graph = workflow.compile(
    checkpointer=memory,
    interrupt_before=["Human_Approval"]
)

In [ ]:
config = {"configurable": {"thread_id": "1"}}
initial_state = {
    "topic": "a localized E. coli outbreak",
    "research_notes": "",
    "current_draft": "",
    "reviewer_feedback": "",
    "approved": False
}

for event in graph.stream(initial_state, config, stream_mode="values"):
    pass  # runs Researcher -> Writer -> Reviewer, then pauses

print("--- PAUSED FOR HUMAN REVIEW ---\n")
print(graph.get_state(config).values["current_draft"])

--- PAUSED FOR HUMAN REVIEW ---

**U.S. Department of Health and Human Services**
**Food and Drug Administration (FDA) / Centers for Disease Control and Prevention (CDC)**

**PUBLIC HEALTH ADVISORY**

**TO:** Healthcare Providers, Public Health Officials, Green Valley County Residents, Media Outlets
**FROM:** U.S. Food and Drug Administration (FDA) and Centers for Disease Control and Prevention (CDC)
**DATE:** [Current Date]
**SUBJECT:** Public Health Advisory: Localized *E. coli* O157:H7 Outbreak in Green Valley County Linked to Unpasteurized Apple Cider

---

**I. Purpose and Summary**

The U.S. Food and Drug Administration (FDA) and Centers for Disease Control and Prevention (CDC), in collaboration with state and local health departments, are actively investigating a localized outbreak of Shiga toxin-producing *Escherichia coli* (STEC) O157:H7 primarily affecting Green Valley County. Epidemiological and laboratory evidence strongly implicates **unpasteurized apple cider sold at "Orc

In [ ]:
while True:
    current_draft = graph.get_state(config).values["current_draft"]
    print("--- CURRENT DRAFT ---\n")
    print(current_draft)
    print("\n----------------------\n")

    user_feedback = input("Type 'approve' to finalize, or give steering instructions (e.g. 'make it more urgent'): ")

    if user_feedback.strip().lower() == "approve":
        graph.update_state(config, {"approved": True, "current_draft": current_draft})
        break
    else:
        revised_prompt = f"""Revise this draft based on the feedback: '{user_feedback}'

Draft:
{current_draft}"""
        revised = llm.invoke(revised_prompt)
        graph.update_state(config, {"current_draft": revised.content})

for event in graph.stream(None, config, stream_mode="values"):
    pass

print("--- FINAL APPROVED STATE ---\n")
print(graph.get_state(config).values["current_draft"])

--- CURRENT DRAFT ---

**U.S. Department of Health and Human Services**
**Food and Drug Administration (FDA) / Centers for Disease Control and Prevention (CDC)**

**PUBLIC HEALTH ADVISORY**

**TO:** Healthcare Providers, Public Health Officials, Green Valley County Residents, Media Outlets
**FROM:** U.S. Food and Drug Administration (FDA) and Centers for Disease Control and Prevention (CDC)
**DATE:** [Current Date]
**SUBJECT:** Public Health Advisory: Localized *E. coli* O157:H7 Outbreak in Green Valley County Linked to Unpasteurized Apple Cider

---

**I. Purpose and Summary**

The U.S. Food and Drug Administration (FDA) and Centers for Disease Control and Prevention (CDC), in collaboration with state and local health departments, are actively investigating a localized outbreak of Shiga toxin-producing *Escherichia coli* (STEC) O157:H7 primarily affecting Green Valley County. Epidemiological and laboratory evidence strongly implicates **unpasteurized apple cider sold at "Orchard Fresh